In [13]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak, Flowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

THRESHOLD_OCCUPANCY = 0.5
THRESHOLD_dSASA = 0.2

out_dir = Path("report")
out_dir.mkdir(parents=True, exist_ok=True)

complex_configs = [
    ('ubl', 'TIM-NTD_x_UHRF1-UBL'),
    ('sra', 'TIM-NTD_x_UHRF1-SRA')
]

data_summaries = {
    f"{k}_{t}": pd.read_csv(Path(p) / 'interface_summary' / f"{p}{s}", sep='\t')
    for k, p in complex_configs
    for t, s in [('pair_frequencies', '_pair_frequencies.tsv'), ('residue_frequencies', '_residue_frequencies.tsv')]
}

res_cols = ['chain', 'resid', 'resname', 'occ_contact', 'mean_rel_dsasa']
pair_cols = ['chain_a', 'resid_a', 'resname_a', 'chain_b', 'resid_b', 'resname_b', 'occ_contact', 'mean_rel_dsasa']

ubl_res_freq = data_summaries['ubl_residue_frequencies'][res_cols].drop_duplicates()
ubl_pair_freq = data_summaries['ubl_pair_frequencies'][pair_cols].drop_duplicates()
sra_res_freq = data_summaries['sra_residue_frequencies'][res_cols].drop_duplicates()
sra_pair_freq = data_summaries['sra_pair_frequencies'][pair_cols].drop_duplicates()

filtered_ubl_res = ubl_res_freq[
    (ubl_res_freq['occ_contact'] >= THRESHOLD_OCCUPANCY) & 
    (ubl_res_freq['mean_rel_dsasa'] >= THRESHOLD_dSASA)
]
filtered_sra_res = sra_res_freq[
    (sra_res_freq['occ_contact'] >= THRESHOLD_OCCUPANCY) & 
    (sra_res_freq['mean_rel_dsasa'] >= THRESHOLD_dSASA)
]

ubl_passed_res_set = set(zip(filtered_ubl_res['chain'], filtered_ubl_res['resid']))
sra_passed_res_set = set(zip(filtered_sra_res['chain'], filtered_sra_res['resid']))

mask_ubl = (
    pd.Series(list(zip(ubl_pair_freq['chain_a'], ubl_pair_freq['resid_a']))).isin(ubl_passed_res_set) |
    pd.Series(list(zip(ubl_pair_freq['chain_b'], ubl_pair_freq['resid_b']))).isin(ubl_passed_res_set)
)
filtered_ubl_pair = ubl_pair_freq[mask_ubl]

mask_sra = (
    pd.Series(list(zip(sra_pair_freq['chain_a'], sra_pair_freq['resid_a']))).isin(sra_passed_res_set) |
    pd.Series(list(zip(sra_pair_freq['chain_b'], sra_pair_freq['resid_b']))).isin(sra_passed_res_set)
)
filtered_sra_pair = sra_pair_freq[mask_sra]

datasets_to_save = [
    ('ubl', filtered_ubl_res, filtered_ubl_pair, 
     filtered_ubl_pair.sort_values(by=['occ_contact', 'mean_rel_dsasa'], ascending=[False, False]).head(20)),
    ('sra', filtered_sra_res, filtered_sra_pair, 
     filtered_sra_pair.sort_values(by=['occ_contact', 'mean_rel_dsasa'], ascending=[False, False]).head(20))
]

for sys_key, f_res, f_pair, top20_df in datasets_to_save:
    f_res.to_csv(out_dir / f"{sys_key}_filtered_residues.csv", index=False)
    f_pair.to_csv(out_dir / f"{sys_key}_filtered_pairs.csv", index=False)
    top20_df.to_csv(out_dir / f"{sys_key}_top20_pairs.csv", index=False)

sns.set_theme(style="whitegrid")

aa_class_map = {
    'ALA': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'VAL': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'LEU': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'ILE': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'MET': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'PRO': 'Hydrophobic (Ala, Val, Leu, Ile, Met, Pro)',
    'PHE': 'Aromatic (Phe, Trp, Tyr)',
    'TRP': 'Aromatic (Phe, Trp, Tyr)',
    'TYR': 'Aromatic (Phe, Trp, Tyr)',
    'SER': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'THR': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'ASN': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'GLN': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'CYS': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'GLY': 'Polar (Ser, Thr, Asn, Gln, Cys, Gly)',
    'ASP': 'Hydrophilic (Asp, Glu, Lys, Arg, His)',
    'GLU': 'Hydrophilic (Asp, Glu, Lys, Arg, His)',
    'LYS': 'Hydrophilic (Asp, Glu, Lys, Arg, His)',
    'ARG': 'Hydrophilic (Asp, Glu, Lys, Arg, His)',
    'HIS': 'Hydrophilic (Asp, Glu, Lys, Arg, His)'
}

ubl_res_freq['aa_class'] = ubl_res_freq['resname'].map(aa_class_map)
sra_res_freq['aa_class'] = sra_res_freq['resname'].map(aa_class_map)

fig, axes = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
for i, (sys_name, df) in enumerate([('UBL', ubl_res_freq), ('SRA', sra_res_freq)]):
    ax = axes[i]
    sns.scatterplot(data=df, x='mean_rel_dsasa', y='occ_contact', hue='aa_class', alpha=0.8, ax=ax, palette='Set2', s=70)
    ax.axvline(THRESHOLD_dSASA, color='dimgray', linestyle='--', linewidth=1.2, label='dSASA Threshold')
    ax.axhline(THRESHOLD_OCCUPANCY, color='gray', linestyle='--', linewidth=1.2, label='Occupancy Threshold')
    ax.set_title(f'{sys_name} - Residue Interface Space', fontsize=12, fontweight='bold')
    ax.set_xlabel('Mean Relative dSASA', fontsize=11)
    ax.set_ylabel('Contact Occupancy', fontsize=11)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small', frameon=True)
plt.tight_layout()
plot1_path = out_dir / 'plot1_residue_space.png'
plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
plt.close()

ubl_top20 = datasets_to_save[0][3].copy()
ubl_top20['pair_label'] = ubl_top20['resname_a'] + ubl_top20['resid_a'].astype(str) + ' - ' + ubl_top20['resname_b'] + ubl_top20['resid_b'].astype(str)
ubl_melted = ubl_top20.melt(id_vars=['pair_label'], value_vars=['occ_contact', 'mean_rel_dsasa'], var_name='Metric', value_name='Value')
ubl_melted['Metric'] = ubl_melted['Metric'].map({'occ_contact': 'Contact Occupancy', 'mean_rel_dsasa': 'Mean Relative dSASA'})

sra_top20 = datasets_to_save[1][3].copy()
sra_top20['pair_label'] = sra_top20['resname_a'] + sra_top20['resid_a'].astype(str) + ' - ' + sra_top20['resname_b'] + sra_top20['resid_b'].astype(str)
sra_melted = sra_top20.melt(id_vars=['pair_label'], value_vars=['occ_contact', 'mean_rel_dsasa'], var_name='Metric', value_name='Value')
sra_melted['Metric'] = sra_melted['Metric'].map({'occ_contact': 'Contact Occupancy', 'mean_rel_dsasa': 'Mean Relative dSASA'})

plt.figure(figsize=(8, 10))
sns.barplot(data=ubl_melted, x='Value', y='pair_label', hue='Metric', palette='Purples')
plt.title('Top 20 Filtered UBL Contact Pairs (Occupancy & dSASA)', fontsize=13, fontweight='bold')
plt.xlabel('Value', fontsize=11)
plt.ylabel('Residue Pair (Chain A: TIM, Chain B: UHRF1)', fontsize=11)
plt.legend(loc='lower right', title='', fontsize='medium')
plt.tight_layout()
ubl_plot_path = out_dir / 'ubl_top20_plot.png'
plt.savefig(ubl_plot_path, dpi=300)
plt.close()

plt.figure(figsize=(8, 10))
sns.barplot(data=sra_melted, x='Value', y='pair_label', hue='Metric', palette='Greens')
plt.title('Top 20 Filtered SRA Contact Pairs (Occupancy & dSASA)', fontsize=13, fontweight='bold')
plt.xlabel('Value', fontsize=11)
plt.ylabel('Residue Pair (Chain A: TIM, Chain B: UHRF1)', fontsize=11)
plt.legend(loc='lower right', title='', fontsize='medium')
plt.tight_layout()
sra_plot_path = out_dir / 'sra_top20_plot.png'
plt.savefig(sra_plot_path, dpi=300)
plt.close()

pdf_filename = out_dir / "interface_analysis_report.pdf"
doc = SimpleDocTemplate(
    str(pdf_filename),
    pagesize=A4,
    rightMargin=36,
    leftMargin=36,
    topMargin=36,
    bottomMargin=36
)

styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'DocTitle', parent=styles['Heading1'], fontSize=16, leading=20, alignment=1, textColor=colors.HexColor('#234e52')
)
heading_style = ParagraphStyle(
    'SectionHeading', parent=styles['Heading2'], fontSize=12, leading=15, textColor=colors.HexColor('#2d3748'), spaceBefore=8, spaceAfter=6
)
body_style = ParagraphStyle(
    'Body', parent=styles['Normal'], fontSize=9, leading=12, textColor=colors.HexColor('#2d3748')
)
callout_style = ParagraphStyle(
    'Callout', parent=styles['Normal'], fontSize=9, leading=13, textColor=colors.HexColor('#1a202c'), backColor=colors.HexColor('#f7fafc'), borderColor=colors.HexColor('#cbd5e0'), borderWidth=1, borderPadding=8, spaceAfter=10
)

class BookmarkFlowable(Flowable):
    def __init__(self, title, key, level=0):
        super().__init__()
        self.title = title
        self.key = key
        self.level = level
        self.width = 0
        self.height = 0

    def draw(self):
        self.canv.bookmarkPage(self.key)
        self.canv.addOutlineEntry(self.title, self.key, self.level)

story = []

story.append(BookmarkFlowable("Protein Interface Analysis Summary Report", "bookmark_main", 0))
story.append(Paragraph("Protein Interface Analysis Summary Report", title_style))
story.append(Spacer(1, 10))

definitions_text = (
    "<b>System Definitions & Chain Mapping:</b><br/>"
    "• <b>Chain A (Chain 1):</b> Corresponds to <b>TIM</b> (Timeless N-terminal domain, <b>TIM-NTD</b>)<br/>"
    "• <b>Chain B (Chain 2):</b> Corresponds to <b>UHRF1</b> (Ubiquitin-like, with PHD and ring finger domains 1)<br/>"
    "• <b>Analyzed Complexes:</b><br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;- <b>UBL:</b> TIM (Chain A) x UHRF1-UBL (Chain B)<br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;- <b>SRA:</b> TIM (Chain A) x UHRF1-SRA (Chain B)"
)
story.append(Paragraph(definitions_text, callout_style))
story.append(Spacer(1, 6))

story.append(Paragraph("This report compiles interface space scatter distributions and top-scoring contact pairs based on double threshold criteria (Contact Occupancy >= 0.5 and Mean Relative dSASA >= 0.2).", body_style))
story.append(Spacer(1, 12))

story.append(BookmarkFlowable("1. Residue Interface Space Overview", "bookmark_sec1", 0))
story.append(Paragraph("1. Residue Interface Space Overview", heading_style))
story.append(Spacer(1, 4))
story.append(Image(str(plot1_path), width=450, height=520))

story.append(PageBreak())
story.append(BookmarkFlowable("2. Top 20 Filtered UBL Contact Pairs", "bookmark_sec2", 0))
story.append(Paragraph("2. Top 20 Filtered UBL Contact Pairs (Chain A: TIM, Chain B: UHRF1)", heading_style))
story.append(Spacer(1, 4))
story.append(Image(str(ubl_plot_path), width=460, height=575))

story.append(PageBreak())
story.append(BookmarkFlowable("3. Top 20 Filtered SRA Contact Pairs", "bookmark_sec3", 0))
story.append(Paragraph("3. Top 20 Filtered SRA Contact Pairs (Chain A: TIM, Chain B: UHRF1)", heading_style))
story.append(Spacer(1, 4))
story.append(Image(str(sra_plot_path), width=460, height=575))

def create_table_data(df_top):
    table_data = [["Residue Pair Label (Chain A: TIM - Chain B: UHRF1)", "Contact Occupancy", "Mean Relative dSASA"]]
    for _, row in df_top.iterrows():
        table_data.append([
            row['pair_label'],
            f"{row['occ_contact']:.3f}",
            f"{row['mean_rel_dsasa']:.3f}"
        ])
    return table_data

def create_res_table_data(df_res):
    table_data = [["Chain", "Residue ID", "Residue Name", "Contact Occupancy", "Mean Relative dSASA"]]
    for _, row in df_res.iterrows():
        table_data.append([
            str(row['chain']),
            str(row['resid']),
            str(row['resname']),
            f"{row['occ_contact']:.3f}",
            f"{row['mean_rel_dsasa']:.3f}"
        ])
    return table_data

story.append(PageBreak())
story.append(BookmarkFlowable("4. Detailed Data Tables", "bookmark_sec4", 0))
story.append(Paragraph("4. Detailed Data Tables", title_style))
story.append(Spacer(1, 10))

story.append(BookmarkFlowable("UBL Filtered Residue Interface Data", "bookmark_sub_ubl_res", 1))
story.append(Paragraph("UBL Filtered Residue Interface Data", heading_style))
t_ubl_res = Table(create_res_table_data(filtered_ubl_res), colWidths=[80, 80, 100, 110, 130])
t_ubl_res.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#f3e8ff')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.HexColor('#6b21a8')),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ('TOPPADDING', (0,0), (-1,-1), 4),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#cbd5e0'))
]))
story.append(t_ubl_res)

story.append(PageBreak())
story.append(BookmarkFlowable("SRA Filtered Residue Interface Data", "bookmark_sub_sra_res", 1))
story.append(Paragraph("SRA Filtered Residue Interface Data", heading_style))
t_sra_res = Table(create_res_table_data(filtered_sra_res), colWidths=[80, 80, 100, 110, 130])
t_sra_res.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#f0fdf4')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.HexColor('#166534')),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ('TOPPADDING', (0,0), (-1,-1), 4),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#cbd5e0'))
]))
story.append(t_sra_res)

story.append(PageBreak())
story.append(BookmarkFlowable("UBL Top Filtered Contact Pairs Data", "bookmark_sub_ubl_pair", 1))
story.append(Paragraph("UBL Top Filtered Contact Pairs Data", heading_style))
t_ubl = Table(create_table_data(ubl_top20), colWidths=[240, 130, 130])
t_ubl.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#f3e8ff')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.HexColor('#6b21a8')),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ('TOPPADDING', (0,0), (-1,-1), 4),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#cbd5e0'))
]))
story.append(t_ubl)

story.append(PageBreak())
story.append(BookmarkFlowable("SRA Top Filtered Contact Pairs Data", "bookmark_sub_sra_pair", 1))
story.append(Paragraph("SRA Top Filtered Contact Pairs Data", heading_style))
t_sra = Table(create_table_data(sra_top20), colWidths=[240, 130, 130])
t_sra.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#f0fdf4')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.HexColor('#166534')),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ('TOPPADDING', (0,0), (-1,-1), 4),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#cbd5e0'))
]))
story.append(t_sra)

doc.build(story)
print("All files and report successfully generated and saved flatly inside 'report/'!")

All files and report successfully generated and saved flatly inside 'report/'!
